In [2]:
import torch

In [141]:
from transformers import AutoTokenizer, BertModel

In [159]:
#checkpoint = "bert-base-uncased"
checkpoint = "F:\\llm_test3\\bert"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = BertModel.from_pretrained(checkpoint)

Some weights of the model checkpoint at F:\llm_test3\bert were not used when initializing BertModel: ['cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [161]:
inputs = tokenizer("Hello, my dog is cute", return_tensors="pt")

In [163]:
outputs = model(**inputs)
last_hidden_states = outputs.last_hidden_state
print(last_hidden_states)

tensor([[[-0.1144,  0.1937,  0.1250,  ..., -0.3827,  0.2107,  0.5407],
         [ 0.5308,  0.3207,  0.3665,  ..., -0.0036,  0.7579,  0.0388],
         [-0.4877,  0.8849,  0.4256,  ..., -0.6976,  0.4458,  0.1231],
         ...,
         [-0.7003, -0.1815,  0.3297,  ..., -0.4838,  0.0680,  0.8901],
         [-1.0355, -0.2567, -0.0317,  ...,  0.3197,  0.3999,  0.1795],
         [ 0.6080,  0.2610, -0.3131,  ...,  0.0311, -0.6283, -0.1994]]],
       grad_fn=<NativeLayerNormBackward0>)


In [175]:
from transformers import BertForMaskedLM
model = BertForMaskedLM.from_pretrained(checkpoint)

Some weights of the model checkpoint at F:\llm_test3\bert were not used when initializing BertForMaskedLM: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [179]:

inputs = tokenizer("The capital of China is [MASK].", return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits
mask_token_index = (inputs.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]
print(mask_token_index)
predicted_token_id = logits[0, mask_token_index].argmax(axis=-1)
tokenizer.decode(predicted_token_id)

tensor([6])


'beijing'

In [195]:
from transformers import BertForNextSentencePrediction
from torch.nn import functional as F

In [185]:
model = BertForNextSentencePrediction.from_pretrained(checkpoint)

Some weights of the model checkpoint at F:\llm_test3\bert were not used when initializing BertForNextSentencePrediction: ['cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertForNextSentencePrediction from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForNextSentencePrediction from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [227]:
prompt = "i am from China"
next_sentence = "Attention is all you need"
encoding = tokenizer(prompt, next_sentence, return_tensors="pt")
outputs = model(**encoding, labels=torch.LongTensor([1]))
softmax = F.softmax(outputs.logits, dim=1)
print(softmax)

tensor([[0.0461, 0.9539]], grad_fn=<SoftmaxBackward0>)


In [261]:
prompt = "Using a Transformer network is simple"
next_sentence = "Attention is all you need"
encoding = tokenizer(prompt, next_sentence, return_tensors="pt")
outputs = model(**encoding, labels=torch.LongTensor([1]))
softmax = F.softmax(outputs.logits, dim=1)
print(softmax)

tensor([[0.9980, 0.0020]], grad_fn=<SoftmaxBackward0>)


In [263]:
encoding.tokens

<bound method BatchEncoding.tokens of {'input_ids': tensor([[  101,  2478,  1037, 10938,  2121,  2897,  2003,  3722,   102,  3086,
          2003,  2035,  2017,  2342,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}>

In [265]:
tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])

['[CLS]',
 'using',
 'a',
 'transform',
 '##er',
 'network',
 'is',
 'simple',
 '[SEP]',
 'attention',
 'is',
 'all',
 'you',
 'need',
 '[SEP]']